# H(12,4): busca global independente
Use outro ambiente Colab com GPU própria; mantenha a busca anterior no ambiente original.
Este notebook testa as 43 variáveis originais, sem assumir as cinco leis iniciais. O espaço é 2^43, 128 vezes maior. A execução possui limite de tempo, podendo terminar incompleta.
A versão nova passou nos controles CPU; a validação CPU/GPU é executada antes da busca.

In [ ]:
from pathlib import Path
Path("RODAR_H12_A100_GLOBAL.py").write_text('"""Colab/Linux: python RODAR_H12_A100_GLOBAL.py (piloto de 2^22).\nVarredura completa: acrescente --full. Validacao local: --validate-only.\nBusca nas 43 variaveis originais, sem assumir as cinco leis iniciais.\nExaustao global exige cobrir exatamente [0,2^43) e Walsh completo dos sobreviventes.\nSomente biblioteca padrao Python + nvcc/CUDA no computador de execucao.\n"""\nimport argparse\nfrom collections import Counter\nimport hashlib\nfrom itertools import combinations\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport time\n\n\n\n\ndef nullspace(rows, n):\n    piv = {}\n    for v in rows:\n        while v:\n            p = v.bit_length()-1\n            if p in piv:\n                v ^= piv[p]\n            else:\n                piv[p] = v\n                break\n    basis = []\n    for j in range(n):\n        if j in piv:\n            continue\n        v = 1 << j\n        for p in sorted(piv):\n            if (v & piv[p]).bit_count() & 1:\n                v ^= 1 << p\n        basis.append(v)\n    assert all(not ((v&r).bit_count()&1) for v in basis for r in rows)\n    return basis\n\n\ndef orbits(k):\n    result, seen = [], set()\n    for c in combinations(range(12), k):\n        m = sum(1 << i for i in c)\n        if m in seen:\n            continue\n        orb = {((m << r) | (m >> (12-r))) & 4095 for r in range(12)}\n        seen.update(orb)\n        result.append(sorted(orb))\n    return result\n\n\ndef xor_selected(z, columns):\n    v = 0\n    while z:\n        bit = z & -z\n        v ^= columns[bit.bit_length()-1]\n        z ^= bit\n    return v\n\n\ndef prepare():\n    orb = orbits(4)\n    assert len(orb) == 43\n    basis = [1 << j for j in range(43)]\n    assert len(basis) == 43\n    original = [sum((sum((x&m)==m for m in o)&1) << x for x in range(4096)) for o in orb]\n    truth = [xor_selected(v, original) for v in basis]\n    filters = [(o[0],1) for o in orbits(8)]\n    masks = [sum(1 << x for x in range(4096) if (x&u)==x) for u,b in filters]\n    # A bit-b of a sum of linear Boolean forms is quadratic if every\n    # nonzero signature multiplicity is divisible by 2^(b-1).\n    # Check that hypothesis on every subcube before interpolating.\n    signatures = [sum(((t >> x)&1) << j for j,t in enumerate(truth)) for x in range(4096)]\n    for u,b in filters:\n        counts = Counter(signatures[x] for x in range(4096) if (x&u)==x)\n        assert all(s==0 or n % (1 << (b-1)) == 0 for s,n in counts.items())\n    def exact(z):\n        t = xor_selected(z, truth)\n        return sum((((t&m).bit_count() >> b)&1) << e for e,((u,b),m) in enumerate(zip(filters,masks)))\n    pairs = list(combinations(range(43),2))\n    v0 = exact(0)\n    singles = [exact(1 << i) for i in range(43)]\n    doubles = [exact((1 << i)|(1 << j)) for i,j in pairs]\n    eqs = []\n    for e,(u,b) in enumerate(filters):\n        c = (v0 >> e)&1\n        lin = sum((((s >> e)&1)^c) << i for i,s in enumerate(singles))\n        rows = [0]*43\n        for (i,j),s in zip(pairs,doubles):\n            if ((s >> e)&1)^c^((lin >> i)&1)^((lin >> j)&1):\n                rows[i] |= 1 << j\n        if c or lin or any(rows):\n            eqs.append((e,c,lin,rows))\n    assert len(eqs) == 43\n    def evaluate(z):\n        values = 0\n        for k,(e,c,lin,rows) in enumerate(eqs):\n            v = c ^ ((z&lin).bit_count()&1)\n            for i in range(43):\n                if (z >> i)&1:\n                    v ^= (z&rows[i]).bit_count()&1\n            values |= v << k\n        return values\n    rng = random.Random(106)\n    checks = [0, (1<<43)-1] + [1 << i for i in range(43)] + [rng.getrandbits(43) for _ in range(1024)]\n    for k in range(8):\n        start = (k*((1<<43)-128))//7\n        checks += list(range(start,start+128))\n    for z in checks:\n        raw = exact(z)\n        expected = sum(((raw >> e)&1) << k for k,(e,*_) in enumerate(eqs))\n        assert evaluate(z) == expected\n    return basis,truth,eqs,checks,[exact(z) for z in checks]\n\n\nCUDA = r\'\'\'\n#include <cuda_runtime.h>\n#include <cstdio>\n#include <cstdlib>\n#include <cstdint>\n#include <algorithm>\n#include <chrono>\n#define CK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){fprintf(stderr,"CUDA ERROR line %d: %s\\n",__LINE__,cudaGetErrorString(e));exit(2);} } while(0)\nusing U = unsigned long long;\n__constant__ U L[43], Q[43][43], T[43][64], C;\n__host__ __device__ inline unsigned parity(U x) {\n#ifdef __CUDA_ARCH__\n return __popcll(x)&1;\n#else\n return __builtin_popcountll(x)&1;\n#endif\n}\n__device__ U values(U x) {\n U result=0;\n for(int e=0;e<43;e++) {\n  U q=0, bits=x;\n  while(bits){int i=__ffsll((long long)bits)-1; q^=Q[e][i]&x; bits&=bits-1;}\n  result |= U(parity(q)^parity(L[e]&x)^((C>>e)&1))<<e;\n }\n return result;\n}\n__device__ bool passes(U x) {\n for(int e=0;e<43;e++) {\n  U q=0,bits=x;\n  while(bits){int i=__ffsll((long long)bits)-1;q^=Q[e][i]&x;bits&=bits-1;}\n  if(parity(q)^parity(L[e]&x)^((C>>e)&1))return false;\n }\n return true;\n}\n__global__ void validate(const U* input,U* output,unsigned n){\n unsigned i=blockIdx.x*blockDim.x+threadIdx.x;\n if(i<n)output[i]=values(input[i]);\n}\n__global__ void scan(U start,unsigned n,U* found,unsigned* count,U* raw){\n U i=U(blockIdx.x)*blockDim.x+threadIdx.x;\n if(i>=n)return;\n U x=start+i;\n if(!passes(x))return;\n atomicAdd(raw,1ULL);\n unsigned weight=0;\n for(int w=0;w<64;w++){\n  U value=0,bits=x;\n  while(bits){int j=__ffsll((long long)bits)-1;value^=T[j][w];bits&=bits-1;}\n  weight+=__popcll(value);\n }\n // W(0)=4096-2*weight must equal +/-64.\n if(weight!=2016 && weight!=2080)return;\n unsigned p=atomicAdd(count,1u);\n // capacity is n, so no candidate can be truncated.\n found[p]=x;\n}\n// DATA\nint main(int argc,char**argv){\n U start=argc>1?strtoull(argv[1],nullptr,0):0;\n U amount=argc>2?strtoull(argv[2],nullptr,0):(1ULL<<22);\n unsigned chunk=1u<<20;\n double limit=argc>3?atof(argv[3]):2400.0;\n auto wall_start=std::chrono::steady_clock::now();U completed=0;\n if(start>(1ULL<<43)||amount>(1ULL<<43)-start){fprintf(stderr,"Invalid range\\n");return 3;}\n CK(cudaSetDevice(0));cudaDeviceProp prop;CK(cudaGetDeviceProperties(&prop,0));\n printf("GPU %s; FULL H(12,4), 43 original variables, no five-law assumption\\n",prop.name);fflush(stdout);\n CK(cudaMemcpyToSymbol(L,hL,sizeof(hL)));CK(cudaMemcpyToSymbol(Q,hQ,sizeof(hQ)));\n CK(cudaMemcpyToSymbol(T,hT,sizeof(hT)));CK(cudaMemcpyToSymbol(C,&hC,sizeof(hC)));\n U *di,*dout;CK(cudaMalloc(&di,sizeof(tests)));CK(cudaMalloc(&dout,sizeof(tests)));\n CK(cudaMemcpy(di,tests,sizeof(tests),cudaMemcpyHostToDevice));\n validate<<<(NTEST+255)/256,256>>>(di,dout,NTEST);CK(cudaGetLastError());CK(cudaDeviceSynchronize());\n U actual[NTEST];CK(cudaMemcpy(actual,dout,sizeof(actual),cudaMemcpyDeviceToHost));\n for(int i=0;i<NTEST;i++)if(actual[i]!=expected[i]){fprintf(stderr,"CPU/GPU validation FAILED at %d\\n",i);return 4;}\n CK(cudaFree(di));CK(cudaFree(dout));printf("CPU_GPU_VALIDATION_PASS %d\\n",NTEST);fflush(stdout);\n U *found,*raw;unsigned *count;\n CK(cudaMalloc(&found,sizeof(U)*chunk));CK(cudaMalloc(&raw,sizeof(U)));CK(cudaMalloc(&count,sizeof(unsigned)));\n U *host=new U[chunk];U totalRaw=0,totalW0=0,totalBent=0;\n cudaEvent_t a,b;CK(cudaEventCreate(&a));CK(cudaEventCreate(&b));\n for(U offset=0;offset<amount;){\n  unsigned n=(unsigned)std::min(U(chunk),amount-offset);\n  CK(cudaMemset(raw,0,sizeof(U)));CK(cudaMemset(count,0,sizeof(unsigned)));\n  CK(cudaEventRecord(a));scan<<<(n+255)/256,256>>>(start+offset,n,found,count,raw);\n  CK(cudaGetLastError());CK(cudaDeviceSynchronize());CK(cudaEventRecord(b));CK(cudaEventSynchronize(b));\n  float ms;CK(cudaEventElapsedTime(&ms,a,b));unsigned hc;U hr;\n  CK(cudaMemcpy(&hc,count,sizeof(hc),cudaMemcpyDeviceToHost));CK(cudaMemcpy(&hr,raw,sizeof(hr),cudaMemcpyDeviceToHost));\n  if(hc>n){fprintf(stderr,"Corrupt count\\n");return 5;}\n  CK(cudaMemcpy(host,found,sizeof(U)*hc,cudaMemcpyDeviceToHost));\n  for(unsigned k=0;k<hc;k++){\n   U z=host[k],coeff=0;U word[64]={0};\n   for(int j=0;j<43;j++)if((z>>j)&1){coeff^=basis[j];for(int w=0;w<64;w++)word[w]^=hT[j][w];}\n   int walsh[4096];for(int x=0;x<4096;x++)walsh[x]=((word[x/64]>>(x%64))&1)?-1:1;\n   for(int step=1;step<4096;step*=2)for(int p=0;p<4096;p+=2*step)for(int q=0;q<step;q++){\n    int u=walsh[p+q],v=walsh[p+q+step];walsh[p+q]=u+v;walsh[p+q+step]=u-v;\n   }\n   bool bent=true;for(int x=0;x<4096;x++)if(abs(walsh[x])!=64){bent=false;break;}\n   if(bent){totalBent++;printf("{\\"bent\\":true,\\"z\\":%llu,\\"coefficients_hex\\":\\"%llx\\"}\\n",z,coeff);}\n  }\n  totalRaw+=hr;totalW0+=hc;offset+=n;completed=offset;\n  printf("{\\"end_exclusive\\":%llu,\\"tested\\":%llu,\\"quadratic_survivors\\":%llu,\\"w0_survivors\\":%llu,\\"bent\\":%llu,\\"gpu_batch_ms\\":%.3f}\\n",start+offset,offset,totalRaw,totalW0,totalBent,ms);fflush(stdout);\n  double elapsed=std::chrono::duration<double>(std::chrono::steady_clock::now()-wall_start).count();\n  if(limit>0 && elapsed>=limit)break;\n }\n printf("{\\"status\\":\\"%s\\",\\"start\\":%llu,\\"count\\":%llu,\\"requested\\":%llu,\\"bent\\":%llu}\\n",completed==amount?"COMPLETE_RANGE":"PARTIAL_TIME_LIMIT",start,completed,amount,totalBent);fflush(stdout);\n CK(cudaFree(found));CK(cudaFree(raw));CK(cudaFree(count));CK(cudaEventDestroy(a));CK(cudaEventDestroy(b));delete[]host;\n return 0;\n}\n\'\'\'\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--full\',action=\'store_true\')\n    ap.add_argument(\'--minutes\',type=float,default=40,help=\'Tempo maximo da busca; conferido entre lotes, 0=sem limite\')\n    ap.add_argument(\'--count\',type=int)\n    ap.add_argument(\'--start\',type=int,default=0)\n    ap.add_argument(\'--validate-only\',action=\'store_true\')\n    args=ap.parse_args()\n    if args.minutes < 0 or args.minutes != args.minutes or args.minutes == float(\'inf\'):\n        ap.error(\'--minutes deve ser finito e >=0\')\n    count=args.count if args.count is not None else (1<<43 if args.full else 1<<22)\n    if not 0<=args.start<=1<<43 or not 0<count<=(1<<43)-args.start:\n        ap.error(\'Intervalo invalido; ao retomar, informe --start e --count restantes.\')\n    folder=Path(\'h12_a100_global\');folder.mkdir(exist_ok=True)\n    t=time.perf_counter()\n    basis,truth,eqs,checks,expected=prepare()\n    scalar=lambda x:f\'{x}ULL\'\n    array=lambda xs:\'{\'+\',\'.join(scalar(x) for x in xs)+\'}\'\n    data=\'\\n\'.join([\n        \'static U hC=\'+scalar(sum(c << k for k,(e,c,l,r) in enumerate(eqs)))+\';\',\n        \'static U hL[43]=\'+array([l for e,c,l,r in eqs])+\';\',\n        \'static U hQ[43][43]={\'+\',\'.join(array(r) for e,c,l,r in eqs)+\'};\',\n        \'static U hT[43][64]={\'+\',\'.join(array([(t>>(64*w))&((1<<64)-1) for w in range(64)]) for t in truth)+\'};\',\n        \'static U basis[43]=\'+array(basis)+\';\',\n        f\'constexpr int NTEST={len(checks)};\',\n        \'static U tests[NTEST]=\'+array(checks)+\';\',\n        \'static U expected[NTEST]=\'+array(expected)+\';\'])\n    source=CUDA.replace(\'// DATA\',data)\n    cu=folder/\'solver.cu\';cu.write_text(source,encoding=\'utf-8\')\n    metadata={\'scope\':\'Full H(12,4), all 43 original quartic cyclic orbit coefficients; no five-law assumption.\',\n              \'basis_43_hex\':[hex(v) for v in basis],\'filters\':43,\'active_equations\':43,\n              \'degree_hypothesis\':\'PASS\',\'interpolation_points\':947,\n              \'dense_and_boundary_checks\':len(checks),\n              \'cuda_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n              \'gpu_executed\':False,\'minutes_limit\':args.minutes}\n    (folder/\'model.json\').write_text(json.dumps(metadata,indent=2),encoding=\'utf-8\')\n    print(f\'CPU: grau quadratico certificado; 43 equacoes; {len(checks)} controles PASS; {time.perf_counter()-t:.2f}s.\',flush=True)\n    print(\'ESCOPO GLOBAL H(12,4): 43 coeficientes originais. Sem assumir as cinco leis iniciais. Completude depende do intervalo coberto.\',flush=True)\n    if args.validate_only:return\n    if not shutil.which(\'nvcc\'):\n        raise SystemExit(\'nvcc ausente. Ative um ambiente CUDA; validacao CPU e fonte foram salvas.\')\n    subprocess.run([\'nvidia-smi\'],check=True)\n    binary=folder/\'solver\'\n    subprocess.run([\'nvcc\',\'-O3\',\'-std=c++14\',str(cu),\'-o\',str(binary)],check=True)\n    name=f\'run_{args.start}_{count}_{time.time_ns()}.log\'\n    with (folder/name).open(\'w\',encoding=\'utf-8\') as log:\n        began=time.perf_counter();last_summary=None;batch_lines=0\n        with subprocess.Popen([str(binary.resolve()),str(args.start),str(count),str(args.minutes*60)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:\n            try:\n                for line in proc.stdout:\n                    log.write(line)\n                    show=True\n                    if line.startswith(\'{\'):\n                        record=json.loads(line)\n                        if \'status\' in record:last_summary=record\n                        if \'end_exclusive\' in record:\n                            batch_lines+=1\n                            show=batch_lines==1 or batch_lines%256==0\n                    if show:\n                        print(line,end=\'\',flush=True);log.flush()\n                rc=proc.wait()\n            except BaseException:\n                proc.terminate()\n                try:\n                    proc.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    proc.kill();proc.wait()\n                raise\n    metadata[\'range_summary\']=last_summary\n    metadata.update(gpu_executed=True,process_exit_code=rc,start=args.start,requested_count=count,log=name)\n    (folder/\'model.json\').write_text(json.dumps(metadata,indent=2),encoding=\'utf-8\')\n    if rc:raise SystemExit(f\'Execucao FALHOU (codigo {rc}); nenhum resultado de completude e valido.\')\n    print(f\'Tempo total: {time.perf_counter()-began:.2f}s. Log: {folder/name}\',flush=True)\n    if last_summary is None:raise SystemExit(\'FALHA: resumo final ausente; completude nao demonstrada.\')\n    print(\'Envie model.json e o log. PARTIAL_TIME_LIMIT significa busca incompleta.\',flush=True)\n\n\nif __name__==\'__main__\':\n    main()\n', encoding="utf-8")
Path("AUDITAR_RESULTADO_H12.py").write_text('"""Depois da busca: %run AUDITAR_RESULTADO_H12.py h12_a100_auditado\nConfere cobertura do log, bent por ANF/FWHT e oito intervalos CPU/GPU.\nNao execute em paralelo com a busca. --self-test usa somente CPU.\n"""\nimport argparse\nimport hashlib\nfrom itertools import combinations\nimport json\nfrom pathlib import Path\nimport random\nimport subprocess\n\n\ndef orbit_data(k):\n    seen=set(); data=[]\n    for combo in combinations(range(12),k):\n        support=sum(1<<j for j in combo)\n        if support in seen:continue\n        terms=[];v=support\n        for _ in range(12):\n            if v not in terms:terms.append(v)\n            v=((v<<1)&4095)|(v>>11)\n        seen.update(terms);data.append(terms)\n    return data\n\n\nORB=orbit_data(4)\n\n\ndef truth_from_anf(c):\n    """Mobius sobre coeficientes ANF; independente da avaliacao por subcubos."""\n    a=[0]*4096\n    for j,terms in enumerate(ORB):\n        if c>>j&1:\n            for m in terms:a[m]^=1\n    for i in range(12):\n        bit=1<<i\n        for x in range(4096):\n            if x&bit:a[x]^=a[x^bit]\n    return sum(v<<x for x,v in enumerate(a))\n\n\ndef is_bent(t):\n    w=[1-2*(t>>x&1) for x in range(4096)]\n    size=1\n    while size<4096:\n        for start in range(0,4096,2*size):\n            for i in range(start,start+size):\n                a,b=w[i],w[i+size];w[i],w[i+size]=a+b,a-b\n        size*=2\n    return all(abs(v)==64 for v in w)\n\n\ndef self_test():\n    quadratic=sum((sum(((x>>(2*i))&1)*((x>>(2*i+1))&1) for i in range(6))&1)<<x for x in range(4096))\n    assert is_bent(quadratic)\n    assert not is_bent(0)\n    assert not is_bent(1)\n    assert len(ORB)==43\n    # Direct polynomial check against Mobius at every input for two quartics.\n    for c in (1,(1<<43)-1):\n        t=truth_from_anf(c)\n        for x in range(4096):\n            direct=sum(sum((x&m)==m for m in terms) for j,terms in enumerate(ORB) if c>>j&1)&1\n            assert t>>x&1==direct\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'folder\',nargs=\'?\',default=\'h12_a100_auditado\')\n    ap.add_argument(\'--self-test\',action=\'store_true\')\n    args=ap.parse_args();self_test()\n    if args.self_test:\n        print(\'SELF_TEST_PASS: bent conhecida aceita, zero e impulso rejeitados; ANF/Mobius concordam.\');return\n    folder=Path(args.folder);model=json.loads((folder/\'model.json\').read_text())\n    digest=hashlib.sha256((folder/\'solver.cu\').read_bytes()).hexdigest()\n    assert digest==model[\'cuda_sha256\'],\'Fonte diferente da registrada\'\n    files=sorted(folder.glob(\'run_*.log\'),key=lambda p:p.stat().st_mtime_ns)\n    assert files,\'Nenhum log encontrado\'\n    # Audit the newest invocation, including a deliberately partial time-limited run.\n    log=files[-1];lines=log.read_text().splitlines()\n    assert any(line.startswith(\'CPU_GPU_VALIDATION_PASS \') for line in lines),\'Validacao GPU ausente\'\n    records=[json.loads(line) for line in lines if line.startswith(\'{\')]\n    summaries=[r for r in records if \'status\' in r];assert len(summaries)==1,\'Resumo final ausente/duplicado\'\n    summary=summaries[0]\n    assert summary[\'status\'] in (\'COMPLETE_RANGE\',\'COMPLETE_CONDITIONAL_RANGE\',\'PARTIAL_TIME_LIMIT\')\n    begin=summary[\'start\'];end=begin;last_tested=last_raw=last_w0=last_bent=0\n    checked_bent=0\n    for r in records:\n        if r.get(\'bent\') is True and \'coefficients_hex\' in r:\n            c=int(r[\'coefficients_hex\'],16)\n            assert c < 1<<43 and is_bent(truth_from_anf(c)),\'Bent anunciada nao confirmada\'\n            checked_bent+=1\n        if \'end_exclusive\' not in r:continue\n        assert r[\'end_exclusive\']>end and r[\'tested\']==r[\'end_exclusive\']-begin\n        assert r[\'tested\']-last_tested<=1<<20\n        raw,w0,bent=r[\'quadratic_survivors\'],r[\'w0_survivors\'],r[\'bent\']\n        assert 0<=bent-last_bent<=w0-last_w0<=raw-last_raw<=r[\'tested\']-last_tested\n        end=r[\'end_exclusive\'];last_tested=r[\'tested\'];last_raw=raw;last_w0=w0;last_bent=bent\n    assert end==begin+summary[\'count\'] and last_bent==summary[\'bent\']==checked_bent\n    key=\'basis_43_hex\' if \'basis_43_hex\' in model else \'basis_36_hex\'\n    basis=[int(v,16) for v in model[key]];n=len(basis);assert n in (36,43)\n    assert 0<=begin<end<=1<<n\n    if \'count\' in model:assert model[\'start\']==begin and model[\'count\']==summary[\'count\']\n    if model.get(\'range_summary\') is not None:assert model[\'range_summary\']==summary\n    if summary[\'status\']==\'PARTIAL_TIME_LIMIT\':assert summary[\'count\']<summary[\'requested\']\n    elif \'requested\' in summary:assert summary[\'count\']==summary[\'requested\']\n    columns=[truth_from_anf(v) for v in basis]\n    filters=[(min(o),1) for o in orbit_data(8)]\n    if n==36:filters += [(u,2) for u in (0x3cf,0x5d7,0x6db)]+[(4095,3)]\n    masks=[sum(1<<x for x in range(4096) if (x&u)==x) for u,b in filters]\n    def cpu(start,count):\n        raw=w0=bent=0\n        for z in range(start,start+count):\n            truth=0\n            for j,t in enumerate(columns):\n                if z>>j&1:truth^=t\n            if any(((truth&m).bit_count()>>b)&1 for m,(u,b) in zip(masks,filters)):continue\n            raw+=1\n            if truth.bit_count() not in (2016,2080):continue\n            w0+=1;bent+=is_bent(truth)\n        return raw,w0,bent\n    rng=random.Random(107);width=128;total=1<<n\n    starts=[0,(1<<20)-64,(1<<32)-64,total-width]+[rng.randrange(total-width+1) for _ in range(4)]\n    results=[]\n    for start in starts:\n        expected=cpu(start,width)\n        p=subprocess.run([str((folder/\'solver\').resolve()),str(start),str(width),\'60\'],capture_output=True,text=True,check=True,timeout=120)\n        rr=[json.loads(line) for line in p.stdout.splitlines() if line.startswith(\'{\')]\n        r=next(r for r in reversed(rr) if \'end_exclusive\' in r)\n        assert r[\'tested\']==width and r[\'end_exclusive\']==start+width\n        actual=tuple(r[k] for k in (\'quadratic_survivors\',\'w0_survivors\',\'bent\'))\n        assert actual==expected,(start,expected,actual)\n        results.append({\'start\':start,\'count\':width,\'counts\':list(actual)})\n        print(f\'INTERVAL_CPU_GPU_PASS start={start} count={width}\',flush=True)\n    report={\'verification\':\'PASS\',\'log\':log.name,\'source_sha256\':digest,\'dimension\':n,\'verified_log_start\':begin,\'verified_log_end_exclusive\':end,\n            \'covers_entire_model_domain\':begin==0 and end==total,\'scope\':\'Global H(12,4)\' if n==43 else \'Conditional on five initial laws\',\n            \'independently_confirmed_bent\':checked_bent,\'interval_tests\':results,\n            \'limitation\':\'Sparse replay strengthens implementation checks; it does not independently repeat the entire search.\'}\n    (folder/\'auditoria_independente.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")


Primeiro execute o piloto (2^22 candidatos). Ele deve mostrar CPU_GPU_VALIDATION_PASS 2093 e COMPLETE_RANGE. Se ocorrer erro, preserve a saída e não prossiga.

In [ ]:
%run RODAR_H12_A100_GLOBAL.py


Depois do piloto, execute a busca abaixo. O limite de 40 minutos é verificado entre lotes e pode ser ultrapassado pelo último lote. Preparação e compilação ficam fora desse limite.
COMPLETE_RANGE só significa conclusão global se start=0 e count=8796093022208. PARTIAL_TIME_LIMIT significa busca incompleta.
Se precisar de menos tempo, altere --minutes 40 para o número disponível.

In [ ]:
%run RODAR_H12_A100_GLOBAL.py --full --minutes 40


Após a busca, o auditor confere o log mais recente e compara oito pequenos intervalos por uma construção independente das funções. Ele verifica uma execução; não junta automaticamente logs de retomadas.

In [ ]:
%run AUDITAR_RESULTADO_H12.py h12_a100_global


Baixe os resultados antes de encerrar o ambiente. Envie o ZIP gerado: contém metadados, fonte CUDA, logs e auditoria.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("RESULTADO_H12_GLOBAL", "zip", "h12_a100_global")
files.download("RESULTADO_H12_GLOBAL.zip")
